# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"Dataset Title: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Published: {metadata.get('datePublished', 'Unknown')}")
print(f"Authors: {[a['@id'] for a in metadata['author']]}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's list all record sets (`cr:RecordSet` class) and their fields and columns using their `@id`s.

In [ ]:
# Get record sets info from metadata
record_sets = metadata.get('recordSet', [])
if not record_sets:
    print("No record sets found in metadata.")
else:
    print("Record Sets Overview:")
    for rs in record_sets:
        rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else str(rs)
        rs_obj = dataset.metadata.record_set(rs_id)
        print(f"- Record Set @id: {rs_id}")
        print(f"  Fields:")
        for field in rs_obj.fields():
            print(f"    - Field @id: {field.id} (Name: {getattr(field, 'name', 'N/A')})")
        print(f"  Columns:")
        for col in rs_obj.columns():
            print(f"    - Column @id: {col.id} (Name: {getattr(col, 'name', 'N/A')})")
        print('-----')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For this notebook, we'll pick the primary survey record set (replace with the actual `@id` from the previous overview), and load all available records into a DataFrame.

In [ ]:
# List the record set IDs
record_set_ids = [rs['@id'] for rs in metadata.get('recordSet', []) if isinstance(rs, dict) and '@id' in rs]
dataframes = {}

# Load records from each record set
for record_set_id in record_set_ids:
    print(f"Loading records for Record Set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        print(df.head())
    else:
        print(f"No records found for Record Set {record_set_id}.")

# Choose one record set for deeper analysis (pick first non-empty)
main_rs_id = next(iter(dataframes)) if dataframes else None
if main_rs_id:
    main_df = dataframes[main_rs_id]
    print(f"Using record set {main_rs_id} for subsequent analysis.")
else:
    print("No record sets with data detected.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Example numeric and group fields - replace with actual field names/@ids from column list above
# Here, suppose 'gad7_score' and 'village' are valid columns

if main_rs_id:
    numeric_field = 'gad7_score'  # Replace if necessary with actual column name
    group_field = 'village'       # Replace if necessary with actual column name

    if numeric_field in main_df.columns:
        # Filter out records with low scores
        threshold = 10
        filtered_df = main_df[main_df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        mean = filtered_df[numeric_field].mean()
        std = filtered_df[numeric_field].std()
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a categorical field
        if group_field in main_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print(f"'{group_field}' field not found for grouping.")
    else:
        print(f"'{numeric_field}' field not found in selected DataFrame.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the GAD-7 score and the mean GAD-7 score by village, if available.

In [ ]:
# Visualization section
if main_rs_id and 'gad7_score' in main_df.columns:
    plt.figure(figsize=(8, 4))
    main_df['gad7_score'].hist(bins=15, color='skyblue', edgecolor='black')
    plt.title('Distribution of GAD-7 Score')
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Frequency')
    plt.show()

    # If 'village' exists, plot group means
    if 'village' in main_df.columns:
        mean_scores = main_df.groupby('village')['gad7_score'].mean()
        mean_scores.sort_values(ascending=False).plot(kind='bar', figsize=(10, 5))
        plt.title('Mean GAD-7 Score by Village')
        plt.ylabel('Mean GAD-7 Score')
        plt.xlabel('Village')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded and previewed FAIR² mental health survey dataset from Kilifi County.
- Demonstrated extraction of structured survey data using `mlcroissant`.
- Filtered and normalized GAD-7 anxiety scores, and visualized their distribution; explored group means by location.
- This notebook can be extended for more detailed statistical or machine learning analyses, exploiting Croissant schema's rich metadata and data access capabilities.